In [6]:
pip install ultralytics

  Using cached ultralytics-8.4.154-py3-none-any.whl.metadata (46 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached polars-1.44.2-py3-none-any.whl.metadata (11 kB)
  Using cached nvidia_ml_py-13.610.43-py3-none-any.whl.metadata (9.7 kB)
  Using cached ultralytics_thop-2.1.6-py3-none-any.whl.metadata (13 kB)
  Using cached ultralytics_platform-0.1.46-py3-none-any.whl.metadata (11 kB)
  Using cached polars_runtime_32-1.44.2-cp310-abi3-win_amd64.whl.metadata (1.5 kB)
Using cached ultralytics-8.4.154-py3-none-any.whl (1.4 MB)
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
Using cached nvidia_ml_py-13.610.43-py3-none-any.whl (53 kB)
Using cached polars-1.44.2-py3-none-any.whl (865 kB)
   ---------------------------------------- 0.0/51.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/51.3 MB ? eta -:--:--
   ---------------------------------------- 0.5/51.3 MB 2.4 MB/s eta 0:00:22
    --------------------------------------- 1.0/5

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
pip install opencv-python

  Using cached opencv_python-5.0.0.93-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached opencv_python-5.0.0.93-cp37-abi3-win_amd64.whl (44.0 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
# import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import hashlib
# from ultralytics import YOLO
import torch
from tqdm.auto import tqdm
from urllib.parse import urlparse
import csv
import time
import xml.etree.ElementTree as ET
import requests

In [2]:
print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

False


In [5]:
# raw dataset
raw_data = Path(r'D:\datasets\data-scanner\raw_data')
raw_images = list(raw_data.rglob("*.webp"))
sitemap_xml = Path(r'D:\datasets\data-scanner\raw_data\wines_sitemap_d778a8e06a.xml')

# create dir and download images from site with metadata
data_dir = Path(r'D:\datasets\data-scanner')
download_images = data_dir / 'download_images'
download_meta = data_dir / 'metadata.csv'
ns = {
    "sm": "http://www.sitemaps.org/schemas/sitemap/0.9",
    "image": "http://www.google.com/schemas/sitemap-image/1.1",
}

In [11]:
def parse_sitemap(path):
  tree = ET.parse(path)
  root = tree.getroot()
  data = []

  for url in root.findall("sm:url", ns):
    wine_url = url.find("sm:loc", ns)
    wine_url = wine_url.text.strip()

    lastmod = url.find("sm:lastmod", ns)
    lastmod = lastmod.text.strip() if lastmod is not None else None

    wine_id = urlparse(wine_url).path.rstrip("/").split("/")[-1]
    images = url.findall("image:image", ns)

    for image_idx, image in enumerate(images):
      image_url = image.find("image:loc", ns)
      image_url = image_url.text.strip()

      title = image.find("image:title", ns)
      title = title.text.strip() if title is not None and title.text else None

      caption = image.find("image:caption", ns)
      caption = caption.text.strip() if caption is not None and caption.text else None

      wine_name = title or caption or wine_id

      data.append({
                "wine_id": wine_id,
                "wine_name": wine_name,
                "wine_url": wine_url,
                "image_url": image_url,
                "lastmod": lastmod,
                "image_index": image_idx,
      })

  return data

def get_ext(url):
  path = urlparse(url).path
  extension = Path(path).suffix.lower()
  if not extension:
    extension = ".jpg"
  return extension

def download_image(session: requests.Session, image_url, data_dir):
  response = session.get(image_url, timeout=30, stream=True)
  response.raise_for_status()
  with data_dir.open("wb") as f:
    for chunk in response.iter_content(chunk_size=1024 * 1024):
      if chunk:
        f.write(chunk)
  return 'downloaded'

In [12]:
data_dir.mkdir(parents=True, exist_ok=True)
download_images.mkdir(parents=True, exist_ok=True)

In [13]:
print('parse sitemap')
records = parse_sitemap(sitemap_xml)
print(f'found: {len(records)} images')

parse sitemap
found: 2037 images


In [14]:
records[1]

{'wine_id': 'fioletovyj-rannij-petnat',
 'wine_name': 'Фиолетовый Ранний гусев',
 'wine_url': 'https://vino-svoe.ru/wines/fioletovyj-rannij-petnat',
 'image_url': 'https://api.vino-svoe.ru/v1/img/str-api/1920/1920/resize/uploads/petnat_fioletovyj_no_bg_preview_carve_photos_e4ddd605bc.webp',
 'lastmod': '2026-09-01T11:46:56.000Z',
 'image_index': 0}

In [16]:
session = requests.Session()
session.headers.update({
  "User-Agent": (
    "Mozilla/5.0 (compatible; WineDatasetDownloader/1.0)"
  )
})

metadata = []
for record in tqdm(records, desc="Downloading"):
  image_url = record["image_url"]
  wine_id = record["wine_id"]
  image_index = record["image_index"]
  extension = get_ext(image_url)

  if image_index == 0:
      filename = f"{wine_id}{extension}"
  else:
      filename = f"{wine_id}_{image_index:02d}{extension}"

  image_path = download_images / filename

  try:
    status = download_image(session, image_url, image_path)
    relative_path = image_path.relative_to(data_dir)

  except requests.RequestException as e:
    print(f"\nFailed: {wine_id}\n" f"URL: {image_url}\n" f"Error: {e}")
    status = "failed"
    relative_path = None

  metadata.append({
      "wine_id": wine_id,
      "wine_name": record["wine_name"],
      "wine_url": record["wine_url"],
      "image_url": record["image_url"],
      "image_path": str(relative_path) if relative_path else None,
      "lastmod": record["lastmod"],
      "download_status": status,
  })

  time.sleep(0.1)

metadata

Downloading:   6%|▌         | 118/2037 [00:45<12:23,  2.58it/s]


KeyboardInterrupt: 

In [17]:
fieldnames = ["wine_id", "wine_name", "wine_url", "image_url", "image_path", "lastmod", "download_status"]
with download_meta.open("w", newline="", encoding="utf-8-sig") as f:
  writer = csv.DictWriter(f, fieldnames=fieldnames)
  writer.writeheader()
  writer.writerows(metadata)

In [19]:
meta = pd.read_csv(r'C:\Users\_\Desktop\wine\metadata.csv')
meta.shape

(118, 7)

In [20]:
from PIL import Image

def sha256(path):
    hasher = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            hasher.update(chunk)
    return hasher.hexdigest()

def preprocess(path, data_dir):
    relative_path = path.relative_to(data_dir)
    file_size = path.stat().st_size
    hash_value = sha256(path)
    with Image.open(path) as img:
        width, height = img.size
        has_alpha = "A" in img.getbands()
    return {
      "img_id": hash_value[:16],
      "file_path": str(relative_path),
      "file_size": file_size,
      "hash_sha256": hash_value,
      "width": width,
      "height": height,
      "has_alpha": has_alpha,
    }

In [21]:
rows = []
for path in tqdm(raw_images, desc="Metadata"):
    rows.append(preprocess(path, raw_data))
df = pd.DataFrame(rows)

Metadata: 100%|██████████| 15213/15213 [00:14<00:00, 1079.04it/s]


In [22]:
df.to_csv('preprocess.csv', index=False)

In [6]:
df = pd.read_csv(data_dir / 'preprocess.csv')
df.head(5)

,img_id,file_path,file_size,hash_sha256,width,height,has_alpha
0,c45df88dc7df42ad,0007_99434f8774.webp,2573502,c45df88dc7df42ad6249b6b59480728f3da3b773e8bd65...,6716,4477,False
1,00859979ef08a1e8,0011_8961f2de4f.webp,1358512,00859979ef08a1e86e236b82bb148d423299e24b7241e9...,5375,3578,False
2,06c1ed77fce4d255,0015_c1e37f7ee1.webp,1492200,06c1ed77fce4d25582b13d5255b3d3914ecda75bcd9c73...,4477,4477,False
3,2d96ea5b33ed10bb,00169_dce952d8a3.webp,5214,2d96ea5b33ed10bb374c6c109cc911377522d0b32a7240...,400,400,False
4,8d1aeb2fb03d7626,001_1_4171b0b864.webp,1484528,8d1aeb2fb03d762676a92925c0d62eeb7c8afc52b7591f...,6000,4000,False


In [7]:
df["file_name"] = df["file_path"].apply(lambda x: Path(x).name)
df["is_thumbnail"] = (df["file_name"].str.lower().str.startswith("thumbnail_"))
df["asset_name"] = (df["file_name"].str.replace(r"^thumbnail_", "", regex=True))

In [8]:
full_assets = set(df.loc[~df["is_thumbnail"], "asset_name"])
df["has_full_image"] = (df["asset_name"].isin(full_assets))
df[df["is_thumbnail"]]["has_full_image"].value_counts()

has_full_image
True    6068
Name: count, dtype: int64

In [9]:

orphan_thumbnail = df[df["is_thumbnail"] & ~df["has_full_image"]]
orphan_thumbnail[["file_path", "width", "height", "file_size", "asset_name"]]

,file_path,width,height,file_size,asset_name


In [10]:
df["use_for_pipeline"] = (~df["is_thumbnail"] | (df["is_thumbnail"] & ~df["has_full_image"]))
df_work = df[df["use_for_pipeline"]].copy()

print("Исходных файлов:", len(df))
print("Для pipeline:", len(df_work))

Исходных файлов: 15213
Для pipeline: 9145


In [11]:
df_work["is_exact_duplicate"] = (df_work.duplicated(subset=["hash_sha256"], keep=False))
print(df_work["is_exact_duplicate"].value_counts())

is_exact_duplicate
False    8116
True     1029
Name: count, dtype: int64


In [12]:
hash_counts = (df_work["hash_sha256"].value_counts())
df_work["exact_duplicate_count"] = (df_work["hash_sha256"].map(hash_counts))

duplicate_hashes = (hash_counts[hash_counts > 1].index)
duplicate_group_map = {hash_value: f"dup_{i:05d}"
    for i, hash_value in enumerate(
        duplicate_hashes,
        start=1
    )
}
df_work["duplicate_group"] = (df_work["hash_sha256"].map(duplicate_group_map))
df_work["is_representative"] = (~df_work.duplicated(subset=["hash_sha256"], keep="first"))

In [13]:
df_unique = (df_work[df_work["is_representative"]].copy())
df_unique = df_unique.drop(['is_thumbnail', 'has_full_image', 'use_for_pipeline', 'duplicate_group', 'is_representative', 'is_exact_duplicate', 'exact_duplicate_count'], axis=1)

In [14]:
df_unique

,img_id,file_path,file_size,hash_sha256,width,height,has_alpha,file_name,asset_name
0,c45df88dc7df42ad,0007_99434f8774.webp,2573502,c45df88dc7df42ad6249b6b59480728f3da3b773e8bd65...,6716,4477,False,0007_99434f8774.webp,0007_99434f8774.webp
1,00859979ef08a1e8,0011_8961f2de4f.webp,1358512,00859979ef08a1e86e236b82bb148d423299e24b7241e9...,5375,3578,False,0011_8961f2de4f.webp,0011_8961f2de4f.webp
2,06c1ed77fce4d255,0015_c1e37f7ee1.webp,1492200,06c1ed77fce4d25582b13d5255b3d3914ecda75bcd9c73...,4477,4477,False,0015_c1e37f7ee1.webp,0015_c1e37f7ee1.webp
3,2d96ea5b33ed10bb,00169_dce952d8a3.webp,5214,2d96ea5b33ed10bb374c6c109cc911377522d0b32a7240...,400,400,False,00169_dce952d8a3.webp,00169_dce952d8a3.webp
4,8d1aeb2fb03d7626,001_1_4171b0b864.webp,1484528,8d1aeb2fb03d762676a92925c0d62eeb7c8afc52b7591f...,6000,4000,False,001_1_4171b0b864.webp,001_1_4171b0b864.webp
...,...,...,...,...,...,...,...,...,...
15208,5a4bbf1100ecc45a,Zunkarskie_vina_caf4dba0ae.webp,91708,5a4bbf1100ecc45a2a42b49a3c40a2ae02abf0696d894c...,1280,720,False,Zunkarskie_vina_caf4dba0ae.webp,Zunkarskie_vina_caf4dba0ae.webp
15209,57e4eebcd9f3d2dd,_5ac9949aff.webp,379488,57e4eebcd9f3d2ddff6970b1c192547fad17c6d9f62bb4...,1680,2520,False,_5ac9949aff.webp,_5ac9949aff.webp
15210,5ac0fcbb8a2cc6df,_adbf5f3bdc.webp,134400,5ac0fcbb8a2cc6df1a28db03569b707b10bea7c31d7bc7...,1024,683,False,_adbf5f3bdc.webp,_adbf5f3bdc.webp
15211,58ada2c2fbbddaf3,_aee68f7604.webp,149164,58ada2c2fbbddaf3f103daccb594724eb4438adec4a866...,1457,1216,False,_aee68f7604.webp,_aee68f7604.webp


In [35]:
model = YOLO("yolo26l.pt")
bottle_class = 39
yolo_conf = 0.25
yolo_imgsz = 640
batch = 16
device = 0

In [15]:
df_unique["full_path"] = (df_unique["file_path"].apply(lambda x: data_dir / Path(x)))
print(df_unique["full_path"].head())

0     D:\datasets\data-scanner\0007_99434f8774.webp
1     D:\datasets\data-scanner\0011_8961f2de4f.webp
2     D:\datasets\data-scanner\0015_c1e37f7ee1.webp
3    D:\datasets\data-scanner\00169_dce952d8a3.webp
4    D:\datasets\data-scanner\001_1_4171b0b864.webp
Name: full_path, dtype: object


In [37]:
summary_rows = []
detection_rows = []

In [44]:
chunk_size = 256

for start in tqdm(range(0, len(df_unique), chunk_size), desc="YOLO chunks"):
    chunk_df = df_unique.iloc[start:start + chunk_size]
    chunk_paths = [str(path) for path in chunk_df["full_path"]]

    results = model.predict(
        source=chunk_paths,
        classes=[bottle_class],
        conf=yolo_conf,
        imgsz=yolo_imgsz,
        batch=batch,
        device=device,
        quantize=16,
        verbose=False,
    )

    for (_, row), result in zip(chunk_df.iterrows(), results):
        boxes = result.boxes

        if boxes is None or len(boxes) == 0:
            summary_rows.append({
                "hash_sha256": row["hash_sha256"],
                "is_bottle": False,
                "bottle_count": 0,
                "bottle_conf": 0.0,
                "bottle_x1": None,
                "bottle_y1": None,
                "bottle_x2": None,
                "bottle_y2": None,
            })
            continue

        confs = (boxes.conf.detach().cpu().numpy())
        coords = (boxes.xyxy.detach().cpu().numpy())
        best_idx = int(np.argmax(confs))
        best_conf = float(confs[best_idx])
        best_box = coords[best_idx]
        x1, y1, x2, y2 = (best_box.tolist())
        summary_rows.append({
            "hash_sha256": row["hash_sha256"],
            "is_bottle": True,
            "bottle_count": len(boxes),
            "bottle_conf": best_conf,
            "bottle_x1": int(x1),
            "bottle_y1": int(y1),
            "bottle_x2": int(x2),
            "bottle_y2": int(y2),
        })

        for detection_id, (conf, box) in enumerate(zip(confs, coords)):
            x1, y1, x2, y2 = box.tolist()
            detection_rows.append({
                "hash_sha256": row["hash_sha256"],
                "detection_id": detection_id,
                "confidence": float(conf),
                "x1": int(x1),
                "y1": int(y1),
                "x2": int(x2),
                "y2": int(y2),
                "bbox_width": int(x2 - x1),
                "bbox_height": int(y2 - y1),
                "bbox_area": int((x2 - x1) * (y2 - y1)),
            })

yolo_summary = pd.DataFrame(
    summary_rows
)

yolo_detections = pd.DataFrame(
    detection_rows
)

YOLO chunks: 100%|██████████| 34/34 [22:55<00:00, 40.46s/it]


In [45]:
yolo_detections

,hash_sha256,detection_id,confidence,x1,y1,x2,y2,bbox_width,bbox_height,bbox_area
0,c45df88dc7df42ad6249b6b59480728f3da3b773e8bd65...,0,0.762207,6207,2788,6322,3145,115,356,41184
1,06c1ed77fce4d25582b13d5255b3d3914ecda75bcd9c73...,0,0.866211,4214,2640,4347,3102,132,461,61363
2,c61613b1d26d7fa9bebced6a7277d86e71e330b2c088eb...,0,0.539062,3913,1244,4003,1546,89,301,26942
3,c61613b1d26d7fa9bebced6a7277d86e71e330b2c088eb...,1,0.462891,4488,2493,4567,2756,78,262,20671
4,c61613b1d26d7fa9bebced6a7277d86e71e330b2c088eb...,2,0.294189,3885,3664,3990,4079,105,414,43548
...,...,...,...,...,...,...,...,...,...,...
9089,5a4bbf1100ecc45a2a42b49a3c40a2ae02abf0696d894c...,9,0.409180,604,293,706,667,102,374,38148
9090,58ada2c2fbbddaf3f103daccb594724eb4438adec4a866...,0,0.339844,352,879,375,955,22,75,1710
9091,58ada2c2fbbddaf3f103daccb594724eb4438adec4a866...,1,0.297363,263,878,287,958,23,79,1904
9092,58ada2c2fbbddaf3f103daccb594724eb4438adec4a866...,2,0.278320,324,878,347,958,22,79,1813


In [46]:
yolo_detections.to_csv('yolo_detections.csv', index=False)
yolo_summary.to_csv('yolo_summary.csv', index=False)

In [19]:
yolo_detections = pd.read_csv(data_dir / 'yolo_detections.csv')
yolo_summary = pd.read_csv(data_dir / 'yolo_summary.csv')

In [23]:
yolo_detections

,hash_sha256,detection_id,confidence,x1,y1,x2,y2,bbox_width,bbox_height,bbox_area
0,c45df88dc7df42ad6249b6b59480728f3da3b773e8bd65...,0,0.762207,6207,2788,6322,3145,115,356,41184
1,06c1ed77fce4d25582b13d5255b3d3914ecda75bcd9c73...,0,0.866211,4214,2640,4347,3102,132,461,61363
2,c61613b1d26d7fa9bebced6a7277d86e71e330b2c088eb...,0,0.539062,3913,1244,4003,1546,89,301,26942
3,c61613b1d26d7fa9bebced6a7277d86e71e330b2c088eb...,1,0.462891,4488,2493,4567,2756,78,262,20671
4,c61613b1d26d7fa9bebced6a7277d86e71e330b2c088eb...,2,0.294189,3885,3664,3990,4079,105,414,43548
...,...,...,...,...,...,...,...,...,...,...
9089,5a4bbf1100ecc45a2a42b49a3c40a2ae02abf0696d894c...,9,0.409180,604,293,706,667,102,374,38148
9090,58ada2c2fbbddaf3f103daccb594724eb4438adec4a866...,0,0.339844,352,879,375,955,22,75,1710
9091,58ada2c2fbbddaf3f103daccb594724eb4438adec4a866...,1,0.297363,263,878,287,958,23,79,1904
9092,58ada2c2fbbddaf3f103daccb594724eb4438adec4a866...,2,0.278320,324,878,347,958,22,79,1813


In [46]:
qa_detections["conf_group"] = pd.cut(
    qa_detections["confidence"],
    bins=[0, 0.4, 0.7, 1.0],
    labels=["low", "medium", "high"],
    include_lowest=True
)

print(
    qa_detections["conf_group"]
    .value_counts(dropna=False)
)

conf_group
high      6115
medium    1768
low       1211
Name: count, dtype: int64


In [47]:
sample_sizes = {
    "low": 50,
    "medium": 50,
    "high": 50,
}

qa_samples = []

for group, n in sample_sizes.items():
    group_df = qa_detections[
        qa_detections["conf_group"] == group
    ]

    qa_samples.append(
        group_df.sample(
            n=min(n, len(group_df)),
            random_state=42
        )
    )

qa_samples = pd.concat(qa_samples, ignore_index=True)

In [48]:
qa_detections = qa_detections.merge(
    df_unique[
        [
            "hash_sha256",
            "file_path"
        ]
    ],
    on="hash_sha256",
    how="left"
)

In [49]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt


def show_detection(row, image_dir, max_size=1000):
    path = image_dir / Path(row["file_path"])

    if not path.exists():
        print(f"Файл не найден: {path}")
        return

    img = cv2.imread(str(path), cv2.IMREAD_COLOR)

    if img is None:
        print(f"Не удалось прочитать: {path}")
        return

    h, w = img.shape[:2]

    # bbox в исходных координатах
    x1 = int(row["x1"])
    y1 = int(row["y1"])
    x2 = int(row["x2"])
    y2 = int(row["y2"])

    # уменьшаем только для визуализации
    scale = min(
        1.0,
        max_size / max(w, h)
    )

    if scale < 1.0:
        new_w = int(w * scale)
        new_h = int(h * scale)

        img = cv2.resize(
            img,
            (new_w, new_h),
            interpolation=cv2.INTER_AREA
        )

        x1 = int(x1 * scale)
        y1 = int(y1 * scale)
        x2 = int(x2 * scale)
        y2 = int(y2 * scale)

    # рисуем ДО RGB conversion
    cv2.rectangle(
        img,
        (x1, y1),
        (x2, y2),
        (0, 0, 255),
        3
    )

    img = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2RGB
    )

    fig, ax = plt.subplots(figsize=(6, 8))

    ax.imshow(img)

    ax.set_title(
        f"conf={row['confidence']:.3f} | "
        f"{row['conf_group']}"
    )

    ax.axis("off")

    plt.show()
    plt.close(fig)

    del img

In [51]:
for _, row in qa_detections.head(20).iterrows():
    show_detection(row, raw_data)

KeyError: 'file_path'